# Notebook 04: Deal Scoring
**Objective:** Build a logistic regression model to predict approval likelihood and identify deal characteristics that drive win rate.

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import shap

In [ ]:
deals_df = pd.read_csv('../data/raw/deals.csv')
approvals_df = pd.read_csv('../data/raw/approvals.csv')
outcomes_df = pd.read_csv('../data/raw/outcomes.csv')

deals_df['created_date'] = pd.to_datetime(deals_df['created_date'])
approvals_df['submitted_date'] = pd.to_datetime(approvals_df['submitted_date'])
approvals_df['decision_date'] = pd.to_datetime(approvals_df['decision_date'])
outcomes_df['close_date'] = pd.to_datetime(outcomes_df['close_date'])

con = duckdb.connect()
con.register('deals', deals_df)
con.register('approvals', approvals_df)
con.register('outcomes', outcomes_df)

print(f"deals:     {len(deals_df):,} rows")
print(f"approvals: {len(approvals_df):,} rows")
print(f"outcomes:  {len(outcomes_df):,} rows")

## Model 1: Predict Approval Outcome

In [ ]:
# Join approvals with deals, exclude pending
approval_model = approvals_df.merge(
    deals_df, on='deal_id'
).query("status != 'Pending'").copy()

# Target: 1 = Approved, 0 = Rejected
approval_model['approved'] = (
    approval_model['status'] == 'Approved'
).astype(int)

# Features
approval_model['is_new_business'] = (
    approval_model['deal_type'] == 'New Business'
).astype(int)
approval_model['is_enterprise'] = (
    approval_model['segment'] == 'Enterprise'
).astype(int)
approval_model['is_mid_market'] = (
    approval_model['segment'] == 'Mid-Market'
).astype(int)

features = [
    'discount_pct', 'arr', 'is_new_business',
    'is_enterprise', 'is_mid_market'
]

X = approval_model[features]
y = approval_model['approved']

print(f"Approval model dataset: {len(X):,} rows")
print(f"Approved: {y.sum():,} ({y.mean():.1%})")
print(f"Rejected: {(1-y).sum():,} ({(1-y).mean():.1%})")

## Train Approval Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

print(f"AUC: {auc:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

## Approval Model Feature Importance

In [ ]:
model = pipeline.named_steps['model']
coefficients = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print(f"{'Feature':<20} {'Coefficient':>12}")
print("-" * 34)
for _, row in coefficients.iterrows():
    direction = '+' if row['coefficient'] > 0 else '-'
    print(
        f"{row['feature']:<20} {direction}{abs(row['coefficient']):>10.3f}"
    )

## Approval Model Feature Importance Chart

In [ ]:
coefficients_sorted = coefficients.sort_values('coefficient')
colors = [
    '#EF553B' if c < 0 else '#636EFA'
    for c in coefficients_sorted['coefficient']
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=coefficients_sorted['coefficient'],
    y=coefficients_sorted['feature'],
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.3f}" for v in coefficients_sorted['coefficient']],
    textposition='outside'
))

fig.update_layout(
    title='Approval Model — Feature Coefficients<br><sup>Positive = more likely approved, Negative = more likely rejected</sup>',
    xaxis_title='Coefficient',
    xaxis=dict(range=[-1.2, 0.8]),
    height=450,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/08_approval_model_coefficients.png')

## Model 2: Predict Win Rate

In [ ]:
win_model = deals_df.merge(outcomes_df, on='deal_id').copy()

# Features
win_model['is_new_business'] = (
    win_model['deal_type'] == 'New Business'
).astype(int)
win_model['is_expansion'] = (
    win_model['deal_type'] == 'Expansion'
).astype(int)
win_model['is_enterprise'] = (
    win_model['segment'] == 'Enterprise'
).astype(int)
win_model['is_mid_market'] = (
    win_model['segment'] == 'Mid-Market'
).astype(int)
win_model['is_approved'] = win_model['deal_id'].isin(
    approvals_df[approvals_df['status'] == 'Approved']['deal_id']
).astype(int)

features_win = [
    'discount_pct', 'arr', 'is_new_business', 'is_expansion',
    'is_enterprise', 'is_mid_market', 'is_approved'
]

X_win = win_model[features_win]
y_win = win_model['win_flag']

print(f"Win model dataset: {len(X_win):,} rows")
print(f"Won:  {y_win.sum():,} ({y_win.mean():.1%})")
print(f"Lost: {(1-y_win).sum():,} ({(1-y_win).mean():.1%})")

## Train Win Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_win, y_win, test_size=0.2, random_state=42, stratify=y_win
)

pipeline_win = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline_win.fit(X_train, y_train)

y_pred = pipeline_win.predict(X_test)
y_prob = pipeline_win.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

print(f"AUC: {auc:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=['Lost', 'Won']))

## Win Model Feature Importance

In [ ]:
model_win = pipeline_win.named_steps['model']
coefficients_win = pd.DataFrame({
    'feature': features_win,
    'coefficient': model_win.coef_[0]
}).sort_values('coefficient', ascending=False)

print(f"{'Feature':<20} {'Coefficient':>12}")
print("-" * 34)
for _, row in coefficients_win.iterrows():
    direction = '+' if row['coefficient'] > 0 else '-'
    print(
        f"{row['feature']:<20} {direction}{abs(row['coefficient']):>10.3f}"
    )

## Win Model Feature Importance Chart

In [ ]:
coefficients_win_sorted = coefficients_win.sort_values('coefficient')
colors = [
    '#EF553B' if c < 0 else '#636EFA'
    for c in coefficients_win_sorted['coefficient']
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=coefficients_win_sorted['coefficient'],
    y=coefficients_win_sorted['feature'],
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.3f}" for v in coefficients_win_sorted['coefficient']],
    textposition='outside'
))

fig.update_layout(
    title='Win Model — Feature Coefficients<br><sup>Positive = more likely to win, Negative = less likely to win</sup>',
    xaxis_title='Coefficient',
    xaxis=dict(range=[-0.8, 0.3]),
    height=450,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/09_win_model_coefficients.png')

## Deal Scoring — Approval Probability

In [ ]:
approval_scores = approval_model.copy()
approval_scores['approval_prob'] = pipeline.predict_proba(
    approval_model[features]
)[:, 1]

scored = approval_scores[[
    'deal_id', 'rep_id', 'segment', 'deal_type',
    'arr', 'discount_pct', 'status', 'approval_prob'
]].sort_values('approval_prob', ascending=True)

print("Bottom 10 deals by approval probability:")
print()
print(
    f"{'Deal':<12} {'Rep':<10} {'Segment':<14} {'Type':<16} "
    f"{'ARR':>10} {'Disc':>8} {'Status':<12} {'Appr Prob':>10}"
)
print("-" * 94)
for _, row in scored.head(10).iterrows():
    print(
        f"{row['deal_id']:<12} {row['rep_id']:<10} {row['segment']:<14} "
        f"{row['deal_type']:<16} ${row['arr']:>8,.0f} "
        f"{row['discount_pct']:>7.1%} {row['status']:<12} "
        f"{row['approval_prob']:>9.1%}"
    )

## Key Findings

- Approval model AUC: 0.746 — discount_pct is the strongest rejection driver
- Win model AUC: 0.694 — deal type and segment drive win rate more than discount
- discount_pct has near-zero effect on win probability (coefficient +0.062) once controlling for segment and deal type
- New Business is the hardest deal type to win (strongest negative coefficient)
- Deals requiring approval are slightly harder to win — borderline deals by nature
- Bottom 10 approval probability deals are all SMB with 23-26% discounts, mostly from REP_001, REP_003, REP_004, REP_005
- Several low-probability deals were approved anyway — model captures exceptions correctly